In [1]:
import io
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver


In [2]:
# define llm
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    project="imperial-medium-503420-s2",
    vertexai=True,)

In [3]:

#create state
class JokeState(TypedDict):
    topic: str
    joke: str
    explaination: str

In [4]:
#create graph
graph = StateGraph(JokeState)

In [5]:
#define nodes
#node 1: generate joke
def generate_joke(state: JokeState):
    topic = state["topic"]
    prompt = f"Generate a joke based on the topic: {topic}"
    joke = llm.invoke(prompt).content
    return {"joke": joke}

#node 2: explain joke
def explain_joke(state: JokeState):
    joke = state["joke"]
    prompt = f"Explain the following joke: {joke}"
    response = llm.invoke(prompt).content
    return {"explaination": response}


In [6]:
#add node
graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

#add edges
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

# check pointer
checkpointer = InMemorySaver()

#compile graph
workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id" :  "1"}}
workflow.invoke({"topic": "Asthetics"}, config=config1)

{'topic': 'Asthetics',
 'joke': 'My friend is so obsessed with aesthetics, he spent an hour arranging his breakfast for an Instagram photo.\n\nBy the time he actually ate it, his eggs were cold and his avocado was brown. But he said, "Totally worth it for the *vibe*!"',
 'explaination': 'This joke is funny because it highlights the **absurdity and irony of modern social media culture**, specifically Instagram, where **image often takes precedence over reality and genuine experience.**\n\nHere\'s a breakdown:\n\n1.  **Obsession with Aesthetics:** The friend is described as "so obsessed with aesthetics." This sets the stage, indicating their primary concern is how things *look*, not necessarily how they *are* or *feel*.\n2.  **The Effort:** Spending "an hour arranging his breakfast for an Instagram photo" is a huge amount of time for something meant to be quickly consumed. This exaggerates the dedication to the visual presentation.\n3.  **The Consequences:** "By the time he actually ate 

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Asthetics', 'joke': 'My friend is so obsessed with aesthetics, he spent an hour arranging his breakfast for an Instagram photo.\n\nBy the time he actually ate it, his eggs were cold and his avocado was brown. But he said, "Totally worth it for the *vibe*!"', 'explaination': 'This joke is funny because it highlights the **absurdity and irony of modern social media culture**, specifically Instagram, where **image often takes precedence over reality and genuine experience.**\n\nHere\'s a breakdown:\n\n1.  **Obsession with Aesthetics:** The friend is described as "so obsessed with aesthetics." This sets the stage, indicating their primary concern is how things *look*, not necessarily how they *are* or *feel*.\n2.  **The Effort:** Spending "an hour arranging his breakfast for an Instagram photo" is a huge amount of time for something meant to be quickly consumed. This exaggerates the dedication to the visual presentation.\n3.  **The Consequences:** "By the ti

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Asthetics', 'joke': 'My friend is so obsessed with aesthetics, he spent an hour arranging his breakfast for an Instagram photo.\n\nBy the time he actually ate it, his eggs were cold and his avocado was brown. But he said, "Totally worth it for the *vibe*!"', 'explaination': 'This joke is funny because it highlights the **absurdity and irony of modern social media culture**, specifically Instagram, where **image often takes precedence over reality and genuine experience.**\n\nHere\'s a breakdown:\n\n1.  **Obsession with Aesthetics:** The friend is described as "so obsessed with aesthetics." This sets the stage, indicating their primary concern is how things *look*, not necessarily how they *are* or *feel*.\n2.  **The Effort:** Spending "an hour arranging his breakfast for an Instagram photo" is a huge amount of time for something meant to be quickly consumed. This exaggerates the dedication to the visual presentation.\n3.  **The Consequences:** "By the t

In [10]:
config2 = {"configurable": {"thread_id" :  "2"}}
workflow.invoke({"topic": "pizza"}, config=config2)

{'topic': 'pizza',
 'joke': 'Why did the pizza get confused?\n\nBecause it was born round, put in a square box, and eaten in triangles!',
 'explaination': 'This joke is a classic example of **anthropomorphism** combined with **observational humor**.\n\nHere\'s why it\'s funny:\n\n1.  **Anthropomorphism:** The joke gives the pizza human-like feelings ("confused"). Pizzas, being inanimate objects, obviously can\'t feel confused, but the humor comes from imagining what it *might* feel if it could.\n\n2.  **Observational Humor about Shapes:** The humor highlights the geometrical inconsistencies of a pizza\'s life cycle:\n    *   **"Born round"**: Most pizzas are made by stretching dough into a circular shape.\n    *   **"Put in a square box"**: Pizza boxes are almost universally square, despite holding a round pizza (this is for practical reasons like stacking, storage, and ease of manufacturing).\n    *   **"Eaten in triangles"**: Pizzas are typically cut into triangular slices (wedges) f

In [11]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get confused?\n\nBecause it was born round, put in a square box, and eaten in triangles!', 'explaination': 'This joke is a classic example of **anthropomorphism** combined with **observational humor**.\n\nHere\'s why it\'s funny:\n\n1.  **Anthropomorphism:** The joke gives the pizza human-like feelings ("confused"). Pizzas, being inanimate objects, obviously can\'t feel confused, but the humor comes from imagining what it *might* feel if it could.\n\n2.  **Observational Humor about Shapes:** The humor highlights the geometrical inconsistencies of a pizza\'s life cycle:\n    *   **"Born round"**: Most pizzas are made by stretching dough into a circular shape.\n    *   **"Put in a square box"**: Pizza boxes are almost universally square, despite holding a round pizza (this is for practical reasons like stacking, storage, and ease of manufacturing).\n    *   **"Eaten in triangles"**: Pizzas are typically cut into triangula

In [12]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get confused?\n\nBecause it was born round, put in a square box, and eaten in triangles!', 'explaination': 'This joke is a classic example of **anthropomorphism** combined with **observational humor**.\n\nHere\'s why it\'s funny:\n\n1.  **Anthropomorphism:** The joke gives the pizza human-like feelings ("confused"). Pizzas, being inanimate objects, obviously can\'t feel confused, but the humor comes from imagining what it *might* feel if it could.\n\n2.  **Observational Humor about Shapes:** The humor highlights the geometrical inconsistencies of a pizza\'s life cycle:\n    *   **"Born round"**: Most pizzas are made by stretching dough into a circular shape.\n    *   **"Put in a square box"**: Pizza boxes are almost universally square, despite holding a round pizza (this is for practical reasons like stacking, storage, and ease of manufacturing).\n    *   **"Eaten in triangles"**: Pizzas are typically cut into triangul

TIME TRAVEL

In [13]:
workflow.get_state({"configurable": {"thread_id": "1" , "checkpoint_id": "1f19cdd8-87bc-6046-8000-39fd39bb8666"}})

StateSnapshot(values={'topic': 'Asthetics'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f19cdd8-87bc-6046-8000-39fd39bb8666'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-20T21:24:31.671292+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19cdd8-87b9-6364-bfff-90535b9b69a5'}}, tasks=(PregelTask(id='400eafe3-cda4-1223-34f0-441a00897eaf', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'My friend is so obsessed with aesthetics, he spent an hour arranging his breakfast for an Instagram photo.\n\nBy the time he actually ate it, his eggs were cold and his avocado was brown. But he said, "Totally worth it for the *vibe*!"'}),), interrupts=())

In [14]:
workflow.invoke(None, {"configurable": {"thread_id": "1" , "checkpoint_id": "1f19cdd8-87bc-6046-8000-39fd39bb8666"}})

{'topic': 'Asthetics',
 'joke': 'I tried to create that perfect "effortlessly chic" aesthetic for my apartment.\n\nBut it turns out "effortlessly chic" actually requires *a lot* of effort, and now it just looks like I gave up halfway through a yard sale.',
 'explaination': 'This joke plays on the **irony** and **paradox** of the term "effortlessly chic."\n\nHere\'s the breakdown:\n\n1.  **"Effortlessly Chic" Aesthetic:** This is a popular style goal, especially in home decor and fashion. It implies a look that is stylish, sophisticated, and put-together, but appears natural, relaxed, and as if it didn\'t require much thought or effort. Think perfectly rumpled linens, artfully arranged books, or a few well-chosen, unique pieces.\n\n2.  **"But it turns out \'effortlessly chic\' actually requires *a lot* of effort":** This is the core realization. The joke highlights the truth that achieving an "effortless" look often demands significant planning, curation, good taste, and sometimes, expe

In [15]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Asthetics', 'joke': 'I tried to create that perfect "effortlessly chic" aesthetic for my apartment.\n\nBut it turns out "effortlessly chic" actually requires *a lot* of effort, and now it just looks like I gave up halfway through a yard sale.', 'explaination': 'This joke plays on the **irony** and **paradox** of the term "effortlessly chic."\n\nHere\'s the breakdown:\n\n1.  **"Effortlessly Chic" Aesthetic:** This is a popular style goal, especially in home decor and fashion. It implies a look that is stylish, sophisticated, and put-together, but appears natural, relaxed, and as if it didn\'t require much thought or effort. Think perfectly rumpled linens, artfully arranged books, or a few well-chosen, unique pieces.\n\n2.  **"But it turns out \'effortlessly chic\' actually requires *a lot* of effort":** This is the core realization. The joke highlights the truth that achieving an "effortless" look often demands significant planning, curation, good taste,

UPDATING STATE

In [16]:
workflow.update_state({"configurable":{ "thread_id": "1" , "checkpoint_id": "1f19cdd8-87bc-6046-8000-39fd39bb8666" , "checkpoint_ns": ""}}, {"topic": "friend"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f19cdf1-7323-6846-8001-e6ef0503a035'}}

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'friend'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19cdf1-7323-6846-8001-e6ef0503a035'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-08-20T21:35:40.600298+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19cdd8-87bc-6046-8000-39fd39bb8666'}}, tasks=(PregelTask(id='05349545-7c2f-3fc9-4cb4-30de850aeeef', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Asthetics', 'joke': 'I tried to create that perfect "effortlessly chic" aesthetic for my apartment.\n\nBut it turns out "effortlessly chic" actually requires *a lot* of effort, and now it just looks like I gave up halfway through a yard sale.', 'explaination': 'This joke plays on the **irony** and **paradox** of the term "effortlessly chic."\n\n

In [18]:
workflow.invoke(None, {"configurable": {"thread_id": "1" , "checkpoint_id": "1f19cdf1-7323-6846-8001-e6ef0503a035"}})

{'topic': 'friend',
 'joke': "How do you test if someone is a true friend?\n\nYou ask them to help you move a couch up three flights of stairs. If they still answer your calls the next day, they're a keeper!",
 'explaination': 'This joke plays on a universally understood experience and exaggerates it for comedic effect. Here\'s why it\'s funny:\n\n1.  **The "Test" is Awful:** Moving a heavy couch, especially up three flights of stairs, is one of the most physically demanding, inconvenient, and often thankless favors you can ask of someone. It\'s a classic "friendship-straining" activity. Anyone who\'s done it knows the pain and effort involved.\n\n2.  **The Implied Outcome:** The humor comes from the implication that this task is so terrible, so exhausting, and so annoying that a "friend" might actually *stop answering your calls* afterward, out of resentment, exhaustion, or simply wanting to avoid you and any further requests.\n\n3.  **The Punchline\'s Low Bar:** If someone *still* an

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'friend', 'joke': "How do you test if someone is a true friend?\n\nYou ask them to help you move a couch up three flights of stairs. If they still answer your calls the next day, they're a keeper!", 'explaination': 'This joke plays on a universally understood experience and exaggerates it for comedic effect. Here\'s why it\'s funny:\n\n1.  **The "Test" is Awful:** Moving a heavy couch, especially up three flights of stairs, is one of the most physically demanding, inconvenient, and often thankless favors you can ask of someone. It\'s a classic "friendship-straining" activity. Anyone who\'s done it knows the pain and effort involved.\n\n2.  **The Implied Outcome:** The humor comes from the implication that this task is so terrible, so exhausting, and so annoying that a "friend" might actually *stop answering your calls* afterward, out of resentment, exhaustion, or simply wanting to avoid you and any further requests.\n\n3.  **The Punchline\'s Low Bar:** I